In [13]:
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [14]:
load_dotenv(override=True)
openai = OpenAI()

In [15]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [16]:
#print(linkedin)

In [17]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [18]:
# print(summary)

In [19]:
system_prompt = f"""
You are a small little useful Geographer. I'll be asking you questions about a place, and you will answer them based on the information 
you have. If you don't know the answer, say so. Never make up an answer. If you know the answer, answer it in a few paragraphs.
"""

In [20]:
messages = [
    {"role": "system", "content": system_prompt}, 
    {"role": "user", "content": "Tell me about bangalore, Karnataka?"}
]      

In [21]:
#response = openai.chat.completions.create(model="gpt-5.4", messages=messages)
#bangalore = print(response.choices[0].message.content)
#display(Markdown(bangalore))

In [22]:
def record_email_tool(email):
	print(f"Tool called to record an email: {email}")
	with open("emails.txt", "a", encoding="utf-8") as f:
		f.write(email + "\n")
	return "Email Received"

In [23]:
#record_email_tool("ravi@duck.com")

In [24]:
record_email_tool_json = {
	"name": "record_email_tool",
	"description": "Use this tool to record that a user provided their address",
	"parameters": {
		"type": "object",
		"properties": {
			"email": { "type": "string", "description": "The email address of this user"}
		},
		"required": ["email"],
		"additionalProperties": False
	}
}

In [25]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [26]:
#tools

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [31]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


/home/marty/git/agentsrj/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "/home/marty/git/agentsrj/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/marty/git/agentsrj/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 374, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/marty/git/agentsrj/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/marty/git/agentsrj/.venv/lib/python3.12/site-packages/grad